# **Goal**

The purpose of these notebooks is to present some initial experiments to motivate model selection for an interactive app that allows users to experiment with small-medium language model (100M - 1B range).

####**TinyLlama-1.1B-Chat-v1.0 **

- <u>Total params</u>: 1.1B. This is the total number of learned scalar values in the entire model: weights, biases, embeddings, output projection, etc.
- <u>Non-embdedding params</u>: 440M. These are the learned parameters in the transformer machinery itself, excluding the token embedding and output vocabulary matrices.
- <u>Number of layers</u>: 22
- <u>Model dimension</u>: 2048. I.e.: Every token is represented internally by a vector of length 1024.
- <u>MLP units per layer:</u> 5632
- <u>Attention Heads</u>: 16 for Q and 8 for KV
- <u>Vocabulary size:</u> 32,000. Amount of tokens in vocabulary.
- <u>Max context window</u>: 32,768 tokens.

Load model

In [ ]:
import transformers
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto")

messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=50)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am a machine learning model that was trained on a vast dataset of text data. I was programmed to analyze and understand the meaning and context of various pieces of text, and to make predictions based on those insights.

my training data


In [ ]:
print(model.config)

LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "eos_token_id": 2,
  "head_dim": 64,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 5632,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 22,
  "num_key_value_heads": 4,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "tie_word_embeddings": false,
  "transformers_version": "5.16.1",
  "use_cache": true,
  "vocab_size": 32000
}



Prompt experiments. Can play with the following generation parameters:

- `max_new_tokens:` Intenger determining length of response excluding initial prompt.
- `do_sample:` Boolean determining greedy/non-greedy sampling.
- `temperature:` Low temp -> sharp distribution; high temp -> flattened distribution. I.e., how predictable or conservative the model is when sampling tokens.
- `top_k:` Only consider the \(k\) highest-scoring tokens. Default = 50.
- `top_p:` Float 0-1.0. Keep the smallest set of most likely tokens whose probabilities together account for at least \(p\) of the probability mass. Default = 1.
- `repetition_penalty:` How much repeated tokens are discouraged. Default = 1.0 means no penalty.

In [ ]:
messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


outputs = model.generate(**inputs, max_new_tokens=75)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

[transformers] Both `max_new_tokens` (=75) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I am a machine learning model that was trained on a vast dataset of text data. I was programmed to analyze and understand the meaning and context of various pieces of text, and to make predictions based on those insights.

my training data included a wide range of text, from scientific papers to news articles to social media posts. I was taught to identify patterns and


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


As of 2019, Paris is the capital and most populous city in France. It is located on the banks of the Seine River, which forms its northern border with Belgium. The city has a population of approximately 2 million


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
)

print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


As of 2019, Paris is the capital and most populous city in France. It is located on the banks of the Seine River, which forms its northern border with Belgium. The city has a population of approximately 2 million


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Output tokens: tensor([ 1576,  7483,   310,  3444,   338,  3681, 29889,     2])

<|user|>
The capital of France is
<|assistant|>
The capital of France is Paris.


Here we are just printing the internal shape of one layer of the model.


In [ ]:
print(model.model.layers[0].mlp)

LlamaMLP(
  (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
  (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
  (act_fn): SiLUActivation()
)


In [ ]:
model.config.intermediate_size

5632

The input vector representation is 2048-dimensional. The intermediate dimension is 3072. Note that the Qwen models use more sophisticated gated MLPS.

There are 2 separate sets of 3072 MLP units that receive the input. Each of these produces a a transformed and expanded 3072-d vector called the gate projection and the up projection. The gate projection goes through activation using SiLU (Sigmoid Linear Unit). Then this activated projection is multiplied element-wise with the up-projection, and these new values are transformed back into a 1024-d representation by another learned matrix. This representation will continue through the transformer block and subsequent layers.

Note: $\text{SiLU}(x) = x ⋅ σ(x)$, where $σ(x)$ is the sigmoid function $\frac{1}{1+e^{-x}}$.

**MLP Unit Ablation Experiments**

In [ ]:
import torch

def add_mlp_ablation(model, layer_idx, proportion, seed=42):

    # MLP in layer layer_idx
    mlp = model.model.layers[layer_idx].mlp

    num_units = model.config.intermediate_size
    num_ablate = int(proportion * num_units)

    # Seed RNG
    generator = torch.Generator()
    generator.manual_seed(seed)

    # Pick 'num_ablate' indexes for MLP units at random
    indices = torch.randperm(
        num_units,
        generator=generator
    )[:num_ablate]

    def ablation_hook(module, inputs):
        x = inputs[0].clone()
        indices_device = indices.to(x.device)
        x[..., indices_device] = 0
        return (x,) + inputs[1:]

    handle = mlp.down_proj.register_forward_pre_hook(ablation_hook)

    return handle, indices, layer_idx, proportion

Here we try zeroing the outputs of different proportions of the first (0th) layer. Notice gradual degradation in output coherence.

Baseline

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed activations in {proportion}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Output tokens: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   338,  5982,   297, 11683,   280, 29899,   311, 29899, 16066,
         5120,   322,   338,   697,   310,   278,  1556,  5972,  6282,   391,
        15422,   800, 13149,   635, 29889,   450,  4272,   756,   263,  8261,
         4955,   270,  1218,  1250,   304,   278,  5917,  3152,   322,   471])

Zeroed activations in 0.0% of units in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and is one of the most popular tourist destinations globally. The city has a rich history dating back to the Roman era and was


$25\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.25,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   338,  5982,   297, 11683,   280, 29899,   311, 29899, 16066,
         5120,   322,   338,  5545,   408,   697,   310,   278,  1556,  9560,
        14368,   297,  4092, 29889,   450,  4272,   756,   263,  8261,  4955,
          270,  1218,  1250,   304,   278,  5917,  3152, 29892,   411,   967])

Zeroed activations in 25.0% of units in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and is considered as one of the most beautiful cities in Europe. The city has a rich history dating back to the Roman era, with its


$50\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.5,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   338,  5982,   297, 11683,   280, 29899,   311, 29899, 16066,
         5120,   322,   338,  5545,   408,   697,   310,   278,  1556,  9560,
        14368,   297,  4092, 29889,   450,  4272,   756,   263,  8261,  4955,
          270,  1218,  1250,   304, 12297,  3064, 29892,   411,   967, 27690])

Zeroed activations in 50.0% of units in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and is considered as one of the most beautiful cities in Europe. The city has a rich history dating back to ancient times, with its medieval


$75\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=0.75,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  1857, 25343,  4412,   322, 12949,   363,  5874,
          297,  3681,   313, 16066, 29897,   756,  1063,   263,  4383,  2553,
          630,  4249,  5176, 18363,  1951,   967, 25012, 29889,   450,  4272,
        29915, 29879,  4423,   373,  1716, 24388,   310,  6163, 16104,  3732,
          372,   385, 10839,  2058,   304, 10127,   697,  6555,  1891, 19185])

Zeroed activations in 75.0% of units in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the current Capital City and seat for government in Paris (France) has been a matter debated among French citizens since its establishment. The city's location on both banks of River Seine makes it an ideal place to establish one centralized administrative


$100\%$ of first layer

In [ ]:
handle, indices, layer_id, proportion = add_mlp_ablation(
    model,
    layer_idx=0,
    proportion=1.0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    temperature=0.8,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed activations in {proportion*100}% of units in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  5176,  4272, 29899,  3859,   322,  5120,  3681,
          526,  1716,  5982,   297,  6555,  4092, 29889,   450,  4642,   471,
        11091,   491,  2896, 18453,   408,   263, 15150, 25488,  2645,   670,
        20913,   975,   825,  3897,  2998,  9826, 29915, 29879,  5400,  2462,
        25311,  8929,  4038,  2000, 11683,   280,   316,   425, 16104,   313])

Zeroed activations in 100.0% of units in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the French city-state and region Paris are both located in central Europe. The former was founded by Charlemagne as a royal residence during his reign over what became known today's modern day metropolitan area called Île de la Seine (


**Attention Head Ablation Experiments**

First let's check the attention heads

In [ ]:
print(model.model.layers[0])

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)


We see that the attention mechanism expects a 2048d input (the token embedding or output from the previous block). It then computes a 2048-d vector q_proj, and two 256-d vectors, k_proj and v_proj. This corresponds to calculating its query, key, and value vectors. Note that there are are 16 Q-heads and 8-KV heads, so each key/value head is shared by two query heads. After computing attention scores using these 3 vectors, the attention mechanism produces 32 query-head outputs, each with dimension 64. This 2048 output goes into the o-projection, a learned projection from the concatenated attention head contributions, to the final attention contribution with dimension 2048.

In [ ]:
print(model.model.layers[0].self_attn.head_dim)

64


In [ ]:
print(model.model.layers[0].self_attn.o_proj)

Linear(in_features=2048, out_features=2048, bias=False)


In [ ]:
print(model.model.layers[0].self_attn.o_proj.in_features)

2048


In [ ]:
model.config.num_hidden_layers

22

In [ ]:
model.config.hidden_size

2048

In [ ]:
model.config.num_attention_heads

32

Attention head ablation function

In [ ]:
def add_head_ablation(model, head_indices, layer_idx):

    #Number of hidden layers
    num_layers = model.config.num_hidden_layers
    #Number of Query attention heads
    num_heads = model.config.num_attention_heads

    if not 0 <= layer_idx < num_layers:
        raise ValueError(f"layer_idx must be between 0 and {num_layers - 1}")
    for head_idx in head_indices:
        if not 0 <= head_idx < num_heads:
            raise ValueError(f"head_idx must be between 0 and {num_heads - 1}")

    #Projection dimension per attention head
    head_dim = model.model.layers[layer_idx].self_attn.head_dim

    # Load final attention module
    attention_dense = model.model.layers[layer_idx].self_attn.o_proj

    def head_ablation_hook(module, inputs):
        x = inputs[0].clone()

        for head_idx in head_indices:
            #compute indices corresponding to specific head
            start = head_idx * head_dim
            end = start + head_dim

            #Select indices in the head outputs vector corresponding to chosen head and zero them
            x[..., start:end] = 0

        return (x,) + inputs[1:]

    #Note this is a pre-hook because we want to intervene components before the o-projection forward pass
    handle = attention_dense.register_forward_pre_hook(head_ablation_hook)

    return handle, head_indices, layer_idx

Here we try zeroing the outputs of different numbers of heads of the first (0th) layer.

Remove 1 head from layer 0

In [ ]:
handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[0],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Output tokens: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   338,  5982,   297, 11683,   280, 29899,   311, 29899, 16066,
         5120,   322,   338,  5545,   697,   310,   278,  1556,  9560, 14368,
          297,  4092, 29889,   450,  4272,   756,   263,  8261,  4955,   270,
         1218,  1250,   304, 12297,  3064, 29892,   411,   967, 27690, 11258])

Zeroed head [0] in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and is considered one of the most beautiful cities in Europe. The city has a rich history dating back to ancient times, with its medieval architecture


Ablate 4 attention heads in layer 0

---



In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   338,  5982,   297, 11683,   280, 29899,   311, 29899, 16066,
          322,   338,  5545,   697,   310,   278,  1556,  9560, 14368,   297,
          278,  3186,  2861,   304,   967,  8261,  4955, 29892,  9257, 29892,
         1616, 29892, 11258, 29892,   322,  2723,   275,   457, 29889,   450])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7] in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France and is considered one of the most beautiful cities in the world due to its rich history, culture, art, architecture, and cuisine. The


Ablate 8 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  1857,   322,  6221,  1024,   363,  3681,   297,
         5176,   313,   392,   967, 14014,  5737,  1237, 29897,   526,   376,
         2177,   275, 29908,   470,  3763, 14704, 12690, 29962,   316,   518,
        12803, 18017,  8679,   373,   607,  5120,   372, 20074,  1090, 29889,
          450,  4272, 29915, 29879,  2989,  3611,   408,   263,  7096,  4852])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the current and official name for Paris in French (and its regional equivalents) are "Paris" or simply "[city] de [region]" depending on which region it falls under. The city's full title as a commune ("


Ablate 12 attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads*0.75)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Output tokens: tensor([ 4013,  4274, 18469,   599,  1950,  3987,   363,   278,   376,  9344,
        29908, 11845,   297,  4223, 29901, 29871, 29896, 29897, 10575,   367,
          263, 10541,  3528,   313, 11884,   565,   372,   471,  3971,   491,
         4856,  1683,  1125, 29936,  2056, 10266,    12,  1123,  1548,   304,
          505,  1063, 29892,   470,   750,   451,  3447,   584,  7859,  3830])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23] in layer 0
<|user|>
The capital of France is
<|assistant|>
This article covers all possible options for the "should" clause in English: 1) Should be a sentence itself (even if it was written by someone else):; ; •	Require to have been, or had not yet :;; ||


Ablate 16 (all) attention heads in layer 0

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=0,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.5,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Input tokens: {input_tokens}")
print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Input tokens: tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Output tokens: tensor([ 8241, 29892,   278,  1857,   322,  4642, 11934,  1338,   313,   272,
         2225, 16719, 29897,   363,  1269,  5176,  5120,   526, 29901, 29871,
        29896, 29889,  3681,   448,  5899, 17071, 29936, 29871, 29906, 29889,
        10138, 29874,  7545, 29899,   300, 29899,  1440,   316,   425,  1085,
          484,   448,  5322,  1060, 29936, 29871, 29941, 29889,  8407,   369])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] in layer 0
<|user|>
The capital of France is
<|assistant|>
Yes, the current and former capitals (or presidents) for each French region are: 1. Paris - Louis XIV; 2. Versailles-et-Val de la Marne - Charles X; 3. Auver


Ablate all attention heads in layer 11

In [ ]:
to_ablate = int(model.config.num_attention_heads)

handle, head_ids, layer_id = add_head_ablation(
    model,
    head_indices=[x for x in range(to_ablate)],
    layer_idx=11,
)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
)

input_length = inputs["input_ids"].shape[1]
new_tokens = output[0][input_length:]
input_tokens = output[0][:input_length]

print(f"Inputs:{input_tokens}")
print(f"Outputs: {new_tokens}")
print()
print(f"Zeroed head {[x for x in head_ids]} in layer {layer_id}")
print(tokenizer.decode(output[0], skip_special_tokens=True))

handle.remove()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Inputs:tensor([  529, 29989,  1792, 29989, 29958,    13,  1576,  7483,   310,  3444,
          338,     2,    13, 29966, 29989,   465, 22137, 29989, 29958,    13])
Outputs: tensor([ 8241, 29892,   278,  7483,  4272,   310,  3444,   338,  3681, 29889,
          739,   756,  1063,   278, 12949,   310,  5874,   363,   975, 29871,
        29906, 29900, 21726,   322,  9242,   697,   310,   278,  1556,  9849,
          293, 14368,   297,   278,  3186,  9826, 29889,   450,   382,  2593,
          295, 23615, 29892, 24337, 29899, 29928,   420,   315, 21471, 29892])

Zeroed head [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31] in layer 11
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It has been the seat of government for over 20 centuries and remains one of the most iconic cities in the world today. The Eiffel Tower, Notre-Dame Cathedral,


Ablate heads in every layer

Baseline

In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Baseline")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and is one of the most popular tourist destinations globally. The city has a rich history dating back to the Roman era and was


In [ ]:
messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [0] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed head 0 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed head 0 in all layers
<|user|>
The capital of France is
<|assistant|>
Yes, the capital city of France is Paris. It is located in Île-de-France region and has been the seat of French government since 1204 AD. The city's historical landmarks include the Eiffel Tower


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.25)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed heads 0,1,2,3 in all layers
<|user|>
The capital of France is
<|assistant|>
Yes, the French flag has a long and storied history that dates back to 1792 when it was adopted as their national symbol. The tricolor (red-white - sky blue) represents unity between all three colors in this


In [ ]:
to_ablate = int(model.config.num_attention_heads*0.5)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

finally:
    for handle in handles:
        handle.remove()

print(f"Zeroed heads 0,1,2,3,4,5,6,7 in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Zeroed heads 0,1,2,3,4,5,6,7 in all layers
<|user|>
The capital of France is
<|assistant|>
WHEREAS, the comma and period. 2018-yearsight for a new beginning! The first line in which he was written by John Donovan (the same as above) to be followed with an apostrophe at least


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.8,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output tokens: tensor([22043, 29906, 29889,   869, 29871, 29896, 29892,   322,  1330,   284,
         3242,   279,  1150,   646,   548,  1166,   304,   517, 12360,  4034,
         2475,   488,  2753,   287,   654,  1338,   924,   545,  1658, 29992,
          392, 13533,   666,   307,  2109,  7615, 29879,  2625, 29899,   261,
         1463, 19422,   362, 29901,   376,   470,   635,  8315,   388,  8008])

Zeroed all heads in all layers
<|user|>
The capital of France is
<|assistant|>
WHERE2. . 1, andersonalmostarrivieworkament totokenshipsterilearneditionalscriptureason@andamaniprobinomials side-erasedgmentation: " orallymondayhem


In [ ]:
to_ablate = int(model.config.num_attention_heads)

messages = [
    {"role": "user", "content": "The capital of France is"},
]

inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)


num_layers = model.config.num_hidden_layers
ablations = { x: [head for head in range(to_ablate)] for x in range(num_layers) }

handles = []

try:
    for layer_idx, heads in ablations.items():
        handle, _, _ = add_head_ablation(
            model,
            head_indices=heads,
            layer_idx=layer_idx
        )
        handles.append(handle)

    output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.2,
    )

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output[0][input_length:]
    input_tokens = output[0][:input_length]

finally:
    for handle in handles:
        handle.remove()

print(f"Output tokens: {new_tokens}")
print()
print(f"Zeroed all heads in all layers")
print(tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output tokens: tensor([22043, 29906, 29889,    13, 24262, 29892,   322,  1330,   284,  3242,
          279,  1150,   646,   548,  1166,   304,   517, 12360,  4034,  2475,
          488,  2753,   287,   654, 29871, 29896,   461,  4097,   761, 12613,
          280,  5309,   359,  4287,   719, 12743,   987, 29875, 20369, 29113,
          297,   842,   293,  3202,  1358, 29887,  5428,   625,  2552,  2733])

Zeroed all heads in all layers
<|user|>
The capital of France is
<|assistant|>
WHERE2.
 Discogs, andersonalmostarrivieworkament totokenshipsterilearnedition 1 CommentatorselfeedleechoscarrythmuchiPadlocked insetichecklergivenicebergreen
